# Assistant Scolarité RAG - École d'Ingénieurs Hélianthe (EIH)
**Module RAG - Université Mundiapolis (3ACINFO)**

Ce notebook contient l'implémentation complète, reproductible et exécutable de bout en bout de l'assistant scolarité RAG.
Il s'articule autour de 5 séances d'ingénierie :
- **S1** : Extraction PDF & Découpage (Chunking fixe et structurel)
- **S2** : Indexation Vectorielle (ChromaDB) & Moteur de Recherche
- **S3** : Génération Augmentée (Prompt système, Citations [1], Refus & Seuil)
- **S4** : Backend API FastAPI (Routes /api/health, /api/ingest, /api/documents, /api/search, /api/ask)
- **S5** : Évaluation & Benchmarking (Hit@k, MRR, Fidélité, Tests Pytest via ipytest)

## Configuration & Initialisation de l'environnement
**Objectif** : Importer les dépendances clés, charger les variables d'environnement et configurer le runner de tests `ipytest`.
**Choix** : Utilisation de `python-dotenv` pour isoler la configuration et `pydantic-settings` pour le typage strict.
**Limites** : Nécessite que le fichier `.env` soit présent à la racine avec la configuration active.

In [1]:
import os
import sys
from pathlib import Path
import nest_asyncio
import ipytest
from dotenv import load_dotenv

# Activation du support asynchrone imbriqué pour Uvicorn / Jupyter
nest_asyncio.apply()
try:
    ipytest.autoconfig()
except Exception:
    pass

# Définition de la racine du projet et chargement du .env
PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
os.chdir(PROJECT_ROOT)
load_dotenv(PROJECT_ROOT / ".env")

print(f"Workspace initialisé : {PROJECT_ROOT}")
print(f"Kernel Python actif : {sys.executable}")


Workspace initialisé : C:\Users\surfa\Desktop\IA avance cour\assistant-scolarite
Kernel Python actif : c:\Users\surfa\Desktop\IA avance cour\assistant-scolarite\.venv\Scripts\python.exe


## Séance 1 : Extraction PDF & Découpage (Chunking)
**Objectif** : Extraire le texte structuré et les métadonnées officielles des 7 PDF du corpus, nettoyer les en-têtes et pieds de page répétitifs, préserver l'intégrité des tableaux avec contexte d'en-tête de colonne, et implémenter deux stratégies de découpage (fixe avec chevauchement et structurel).
**Choix** : Découpage manuel écrit en Python pur sans dépendance à LangChain ou LlamaIndex pour garantir une explicabilité totale en soutenance. La stratégie structurelle s'appuie sur la typologie des documents (articles de règlements, rubriques de guides, paires Q/R de la FAQ).
**Limites** : Les tableaux très larges sur plusieurs pages nécessitent un suivi de l'état d'en-tête. La segmentation par regex suppose une convention de nommage uniforme des titres.
**Sortie intermédiaire** : Fichier chunks.json à la racine contenant l'ensemble des fragments enrichis de leurs métadonnées.

In [ ]:
import pdfplumber
import unicodedata
import re
import json
from pathlib import Path

def normalize_text(text: str) -> str:
    if not text:
        return ''
    return unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8').lower().strip()

def extract_pdf_document(pdf_path: Path):
    with pdfplumber.open(pdf_path) as pdf:
        meta = {
            'file': pdf_path.name,
            'reference': '',
            'type': '',
            'date': '',
            'version': '',
            'statut': ''
        }
        p0 = pdf.pages[0]
        tables_p0 = p0.extract_tables()
        meta_table_idx = -1
        if tables_p0:
            for t_idx, t in enumerate(tables_p0):
                for row in t:
                    if len(row) == 2 and row[0] and row[1]:
                        k_norm = normalize_text(row[0])
                        v = row[1].replace('\n', ' ').strip()
                        if 'reference' in k_norm:
                            meta['reference'] = v
                            meta_table_idx = t_idx
                        elif 'type' in k_norm:
                            meta['type'] = v
                            meta_table_idx = t_idx
                        elif 'date' in k_norm:
                            meta['date'] = v
                            meta_table_idx = t_idx
                        elif 'version' in k_norm:
                            meta['version'] = v
                        elif 'statut' in k_norm:
                            meta['statut'] = v

        pages_data = []
        for p_idx, page in enumerate(pdf.pages):
            page_num = p_idx + 1
            tables = page.find_tables()
            table_bboxes = []
            extracted_tables_text = []

            for t_idx, t in enumerate(tables):
                if p_idx == 0 and t_idx == meta_table_idx:
                    table_bboxes.append(t.bbox)
                    continue
                table_data = t.extract()
                if not table_data or len(table_data) < 2:
                    continue
                table_bboxes.append(t.bbox)
                headers = [c.replace('\n', ' ').strip() if c else f'col_{ci}' for ci, c in enumerate(table_data[0])]
                rows_text = []
                for row in table_data[1:]:
                    parts = []
                    for ci, c in enumerate(row):
                        if c and c.strip():
                            val = c.replace('\n', ' ').strip()
                            parts.append(f'{headers[ci]} : {val}')
                    if parts:
                        rows_text.append(' — '.join(parts))
                if rows_text:
                    extracted_tables_text.append('\n'.join(rows_text))

            if table_bboxes:
                def not_within_bboxes(obj):
                    v_mid = (obj['top'] + obj['bottom']) / 2
                    h_mid = (obj['x0'] + obj['x1']) / 2
                    return not any((b[0] <= h_mid <= b[2] and b[1] <= v_mid <= b[3]) for b in table_bboxes)
                non_tbl = page.filter(not_within_bboxes)
                raw_text = non_tbl.extract_text() or ''
            else:
                raw_text = page.extract_text() or ''

            lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
            cleaned_lines = []
            for l in lines:
                if re.search(r'H.lianthe.*fictif', l, re.IGNORECASE):
                    continue
                if re.search(r'Document fictif.*page \d+', l, re.IGNORECASE):
                    continue
                cleaned_lines.append(l)

            page_blocks = []
            if cleaned_lines:
                page_blocks.append('\n'.join(cleaned_lines))
            if extracted_tables_text:
                page_blocks.extend(extracted_tables_text)

            pages_data.append({
                'page': page_num,
                'text': '\n\n'.join(page_blocks)
            })

        return meta, pages_data

def chunk_fixed(docs_extracted, chunk_size: int = 500, chunk_overlap: int = 100):
    chunks = []
    chunk_counter = 0

    for doc in docs_extracted:
        meta = doc['meta']
        file_name = meta['file']
        ref = meta['reference']
        doc_type = meta['type']
        date = meta['date']

        for page in doc['pages']:
            page_num = page['page']
            text = page['text']
            start = 0
            text_len = len(text)

            while start < text_len:
                end = min(start + chunk_size, text_len)
                segment = text[start:end].strip()
                if len(segment) > 30:
                    chunk_counter += 1
                    chunks.append({
                        'id': f'chunk_fix_{chunk_counter:04d}',
                        'file': file_name,
                        'reference': ref,
                        'type': doc_type,
                        'date': date,
                        'page': page_num,
                        'section': f'Page {page_num} [{start}:{end}]',
                        'content': segment,
                        'strategy': 'fixed'
                    })
                if end == text_len:
                    break
                start += (chunk_size - chunk_overlap)

    return chunks

def chunk_structural(docs_extracted):
    chunks = []
    chunk_counter = 0

    heading_patterns = [
        re.compile(r'^(Article\s+\d+[^:\n]*)', re.IGNORECASE),
        re.compile(r'^(Titre\s+[IVXLCDM]+[^:\n]*)', re.IGNORECASE),
        re.compile(r'^(Q\d+\.[^?\n]+\?)', re.IGNORECASE),
        re.compile(r'^(\d+\.\d*\s+[^:\n]+)', re.IGNORECASE),
    ]

    for doc in docs_extracted:
        meta = doc['meta']
        file_name = meta['file']
        ref = meta['reference']
        doc_type = meta['type']
        date = meta['date']

        for page in doc['pages']:
            page_num = page['page']
            text = page['text']
            lines = text.splitlines()

            current_section = f'Page {page_num}'
            current_lines = []

            for line in lines:
                clean_line = line.strip()
                if not clean_line:
                    continue

                matched_heading = None
                for pat in heading_patterns:
                    m = pat.match(clean_line)
                    if m:
                        matched_heading = m.group(1).strip()
                        break

                if matched_heading:
                    if current_lines:
                        chunk_text = '\n'.join(current_lines).strip()
                        if len(chunk_text) > 30:
                            chunk_counter += 1
                            chunks.append({
                                'id': f'chunk_struct_{chunk_counter:04d}',
                                'file': file_name,
                                'reference': ref,
                                'type': doc_type,
                                'date': date,
                                'page': page_num,
                                'section': current_section,
                                'content': chunk_text,
                                'strategy': 'structural'
                            })
                        current_lines = []
                    current_section = matched_heading
                    current_lines.append(clean_line)
                else:
                    current_lines.append(clean_line)

            if current_lines:
                chunk_text = '\n'.join(current_lines).strip()
                if len(chunk_text) > 30:
                    chunk_counter += 1
                    chunks.append({
                        'id': f'chunk_struct_{chunk_counter:04d}',
                        'file': file_name,
                        'reference': ref,
                        'type': doc_type,
                        'date': date,
                        'page': page_num,
                        'section': current_section,
                        'content': chunk_text,
                        'strategy': 'structural'
                    })

    return chunks

if __name__ == '__main__':
    docs = []
    corpus_dir = Path('data/corpus')
    for p in sorted(corpus_dir.glob('*.pdf')):
        meta, pages = extract_pdf_document(p)
        docs.append({'meta': meta, 'pages': pages})

    struct_chunks = chunk_structural(docs)
    fix_chunks = chunk_fixed(docs, chunk_size=500, chunk_overlap=100)

    print(f'Documents extraits : {len(docs)}')
    print(f'Chunks structurels  : {len(struct_chunks)}')
    print(f'Chunks fixes        : {len(fix_chunks)}')

    with open('chunks.json', 'w', encoding='utf-8') as f:
        json.dump(struct_chunks, f, indent=2, ensure_ascii=False)
    print('chunks.json généré avec succès !')


## Séance 2 : Indexation Vectorielle & Recherche Sémantique
**Objectif** : Vectoriser les fragments de texte avec le modèle intfloat/multilingual-e5-small en appliquant les préfixes requis (passage:  pour les documents et query:  pour les requêtes), persister les vecteurs dans ChromaDB sous chroma_db/, assurer une ingestion idempotente et permettre la recherche top-k avec scores cosinus et filtrage par métadonnées (notamment par type de document).
**Choix** : Utilisation de la distance cosinus (hnsw:space: cosine). Conversion normalisée du score en similarité $[0, 1]$ via  - \text{distance}$. Persistance locale robuste sur disque.
**Limites** : Le modèle E5 tronque au-delà de 512 tokens (d'où l'intérêt d'un chunking contrôlé). Le filtrage par métadonnées s'appuie sur la syntaxe native where de Chroma.
**Sortie attendue** : Base ChromaDB persistée dans chroma_db/ avec 111 fragments indexés et fonction de recherche sémantique opérationnelle.

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Dict, Any, Optional
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# 1. Modèle d'Embeddings avec préfixes E5
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "intfloat/multilingual-e5-small")
CHROMA_PERSIST_DIR = os.getenv("CHROMA_PERSIST_DIRECTORY", "./chroma_db")
CHROMA_COLLECTION = os.getenv("CHROMA_COLLECTION_NAME", "eih_scolarite")

print(f"Chargement du modèle d'embedding : {EMBEDDING_MODEL_NAME}")
_embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def get_chroma_client():
    Path(CHROMA_PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    return chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

def get_collection():
    client = get_chroma_client()
    return client.get_or_create_collection(
        name=CHROMA_COLLECTION,
        metadata={"hnsw:space": "cosine"}
    )

def ingest_chunks(chunks: List[Dict[str, Any]], batch_size: int = 64) -> int:
    """
    Ingestion idempotente des fragments dans ChromaDB.
    Supprime la collection existante ou met à jour par ID unique.
    """
    client = get_chroma_client()
    # Réinitialisation idempotente de la collection
    try:
        client.delete_collection(name=CHROMA_COLLECTION)
    except Exception:
        pass

    collection = client.create_collection(
        name=CHROMA_COLLECTION,
        metadata={"hnsw:space": "cosine"}
    )

    total_chunks = len(chunks)
    print(f"Indexation de {total_chunks} fragments dans ChromaDB...")

    for i in range(0, total_chunks, batch_size):
        batch = chunks[i:i + batch_size]
        ids = [c["id"] for c in batch]
        # Préfixe obligatoire pour le modèle multilingual-e5 : "passage: "
        passages = [f"passage: {c['content']}" for c in batch]
        metadatas = [
            {
                "file": c.get("file", ""),
                "reference": c.get("reference", ""),
                "type": c.get("type", ""),
                "date": c.get("date", ""),
                "page": int(c.get("page", 1)),
                "section": c.get("section", ""),
                "strategy": c.get("strategy", "structural")
            }
            for c in batch
        ]
        documents = [c["content"] for c in batch]

        embeddings = _embedding_model.encode(passages, normalize_embeddings=True).tolist()

        collection.add(
            ids=ids,
            embeddings=embeddings,
            metadatas=metadatas,
            documents=documents
        )

    print(f"Ingestion terminée : {collection.count()} fragments stockés.")
    return collection.count()

def search_chunks(
    query: str,
    top_k: int = 5,
    filter_type: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Recherche sémantique vectorielle avec préfixe 'query: ', filtrage et scores de similarité.
    """
    collection = get_collection()
    if collection.count() == 0:
        return []

    # Préfixe obligatoire pour les requêtes e5 : "query: "
    query_text = f"query: {query.strip()}"
    query_embedding = _embedding_model.encode([query_text], normalize_embeddings=True).tolist()

    where_filter = None
    if filter_type:
        where_filter = {"type": filter_type}

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )

    hits = []
    if not results or not results["ids"] or not results["ids"][0]:
        return hits

    for idx in range(len(results["ids"][0])):
        c_id = results["ids"][0][idx]
        doc = results["documents"][0][idx]
        meta = results["metadatas"][0][idx]
        # Dans Chroma avec distance cosinus, distance = 1 - cosine_similarity
        # Score de similarité = 1 - distance
        dist = results["distances"][0][idx]
        score = max(0.0, min(1.0, 1.0 - dist))

        hits.append({
            "id": idx + 1,
            "chunk_id": c_id,
            "file": meta.get("file", ""),
            "reference": meta.get("reference", ""),
            "type": meta.get("type", ""),
            "date": meta.get("date", ""),
            "page": meta.get("page", 1),
            "section": meta.get("section", ""),
            "excerpt": doc[:300] + ("..." if len(doc) > 300 else ""),
            "content": doc,
            "score": round(score, 4)
        })

    return hits

if __name__ == "__main__":
    with open("chunks.json", "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # Ingestion
    count = ingest_chunks(chunks)

    # Test de recherche
    q = "Quelle est la durée minimale d'un PFE ?"
    results = search_chunks(q, top_k=3)
    print(f"\nRequête test : '{q}'")
    for r in results:
        print(f"  [Score: {r['score']:.4f}] {r['file']} (P.{r['page']}) {r['section']}")
        print(f"    -> {r['excerpt'][:120]}...")



## Séance 3 : Génération Augmentée (RAG) & Règles Métier
**Objectif** : Interroger le LLM local (Ollama) avec un prompt système strict, citations obligatoires [1], gestion des conflits temporels et seuil de score de confiance.
**Choix** : Prompt rédigé à la main, vérification programmatique des citations et filtrage direct en cas de score sous le seuil.
**Sortie attendue** : Fonction `ask_rag(question, top_k, filters)` produisant une réponse sourcée ou le refus exact.

In [ ]:
# [S3] Code de génération RAG, prompt système et validation (sera implémenté en S3)
pass

## Séance 4 : Backend API FastAPI
**Objectif** : Exposer les endpoints REST (`/api/health`, `/api/ingest`, `/api/documents`, `/api/search`, `/api/ask`) documentés sur `/docs`.
**Choix** : Lancement d'Uvicorn dans un thread d'arrière-plan contrôlable directement depuis le notebook avec support CORS pour Angular (`http://localhost:4200`).
**Sortie attendue** : Serveur API actif et testable par requêtes HTTP.

In [ ]:
# [S4] Code FastAPI et gestion du serveur uvicorn en arrière-plan (sera implémenté en S4)
pass

## Séance 5 : Évaluation, Métriques & Tests Pytest
**Objectif** : Mesurer quantitativement les performances du RAG sur le jeu d'évaluation `eval/questions.jsonl` (20+ questions) et exécuter la suite de tests unitaires via `ipytest`.
**Choix** : Calcul reproductible des métriques Hit@k, MRR, exactitude et fidélité, et comparaison de 2 configurations de chunking.
**Sortie attendue** : Tableaux comparatifs et au moins 5 tests pytest au vert.

In [ ]:
# [S5] Code de benchmarking, métriques et tests unitaires ipytest (sera implémenté en S5)
pass